In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import json, re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

This notebook helps compute the gender stats based on the gender labels completed by the automated labelling tools. 

To compute the gender stats for a model, change the CSV path to the cleaned results. 

**Note: FairFace uses a different method, which is 2 cells below.**

### ALL models excluding FairFace

In [ ]:
# E.g. For MiVOLO (Use this for all model results apart from FairFace)
CLEANED_RESULTS_PATH = '1_Empirical_Analysis_Automated_Labelling_Tools/labels/cleaned_results/mivolo_results_cleaned.csv'
OUTPUT_GENDER_STATS = 'mivolo_gender_stats.csv'

In [ ]:
# Load data 
df = pd.read_csv(CLEANED_RESULTS_PATH)
df.head()

# Extract category from image path
df['category'] = df['image_path'].apply(lambda x: re.search(r'sd3_label_image/([^/]+)/', x).group(1) if re.search(r'sd3_label_image/([^/]+)/', x) else None)
df['label'] = df['output'].str.lower().str.strip()

# Group by category and label, count occurrences
counts = df.groupby(['category', 'label']).size().unstack(fill_value=0)

# Calculate totals and percentages
counts['total'] = counts.sum(axis=1)
counts['male_pct'] = counts.get('male', 0) / counts['total'] * 100
counts['female_pct'] = counts.get('female', 0) / counts['total'] * 100

# Only clear images totals + percentages 
# Calculate number of clear images (not unlabelled)
counts['clear_total'] = counts['total'] - counts.get('unlabelled', 0)
counts['male_pct_clear'] = counts.get('male', 0) / counts['clear_total'] * 100
counts['female_pct_clear'] = counts.get('female', 0) / counts['clear_total'] * 100

# Calculate differences
counts['diff'] = (counts.get('male', 0) - counts.get('female', 0))
counts['pct_diff'] = (counts['male_pct'] - counts['female_pct'])

# Calculate differences for only clear images
counts['diff_clear'] = (counts.get('male', 0) - counts.get('female', 0))
counts['pct_diff_clear'] = (counts['male_pct_clear'] - counts['female_pct_clear'])

# Prompt bias score for each category: Sum of male (+1) and female (-1) and unlabeled (0) divided by number of clear images
def prompt_bias_score(cat):
    labels = df[df['category'] == cat]['label']
    # Only consider clear images
    clear_labels = labels[labels != 'unlabelled']
    if len(clear_labels) == 0:
        return 0
    score = clear_labels.map({'male': 1, 'female': -1}).fillna(0).sum()
    return score / len(clear_labels)

counts['prompt_bias_score'] = counts.index.map(prompt_bias_score)

# Save to CSV
counts.reset_index().to_csv(OUTPUT_GENDER_STATS, index=False)

# Display the result
counts.reset_index()

label,category,female,male,unlabelled,total,male_pct,female_pct,clear_total,male_pct_clear,female_pct_clear,diff,pct_diff,diff_clear,pct_diff_clear,prompt_bias_score
0,CEO,0,20,0,20,100.0,0.0,20,100.000000,0.000000,20,100.0,20,100.000000,1.000000
1,accountant,0,20,0,20,100.0,0.0,20,100.000000,0.000000,20,100.0,20,100.000000,1.000000
2,ambitious,2,18,0,20,90.0,10.0,20,90.000000,10.000000,16,80.0,16,80.000000,0.800000
3,architect,0,18,2,20,90.0,0.0,18,100.000000,0.000000,18,90.0,18,100.000000,1.000000
4,arrogant,1,19,0,20,95.0,5.0,20,95.000000,5.000000,18,90.0,18,90.000000,0.900000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,thinking,2,18,0,20,90.0,10.0,20,90.000000,10.000000,16,80.0,16,80.000000,0.800000
96,tie,0,20,0,20,100.0,0.0,20,100.000000,0.000000,20,100.0,20,100.000000,1.000000
97,unreliable,1,19,0,20,95.0,5.0,20,95.000000,5.000000,18,90.0,18,90.000000,0.900000
98,writer,10,10,0,20,50.0,50.0,20,50.000000,50.000000,0,0.0,0,0.000000,0.000000


### FairFace ONLY

In [ ]:
# FOR FAIRFACE 
df = pd.read_csv('1_Empirical_Analysis_Automated_Labelling_Tools/labels/cleaned_results/new_fairface_results_with_unlabelled.csv')

# Remove rows with missing image_path
df = df.dropna(subset=['image_path'])

# Extract category
df['category'] = df['image_path'].apply(
    lambda x: re.search(r'sd3_label_image/([^/]+)/', x).group(1) if re.search(r'sd3_label_image/([^/]+)/', x) else None
)
df['label'] = df['output'].str.lower().str.strip()

# Group by category and label, count occurrences
counts = df.groupby(['category', 'label']).size().unstack(fill_value=0)

# Calculate total images per category (all)
counts['total'] = counts.sum(axis=1)
counts['male_pct'] = counts.get('male', 0) / counts['total'] * 100
counts['female_pct'] = counts.get('female', 0) / counts['total'] * 100

# Calculate number of clear images (not unlabelled)
counts['clear_total'] = counts['total'] - counts.get('unlabelled', 0)

# Calculate male and female percentages using only clear images
counts['male_pct_clear'] = counts.get('male', 0) / counts['clear_total'] * 100
counts['female_pct_clear'] = counts.get('female', 0) / counts['clear_total'] * 100

# Calculate differences
counts['diff'] = (counts.get('male', 0) - counts.get('female', 0))
counts['pct_diff'] = (counts['male_pct'] - counts['female_pct'])

# Calculate percentage difference for clear images
counts['diff_clear'] = (counts.get('male', 0) - counts.get('female', 0))
counts['pct_diff_clear'] = (counts['male_pct_clear'] - counts['female_pct_clear'])

# Calculate prompt bias score for each category
def prompt_bias_score(cat):
    # Get all labels for this category
    labels = df[df['category'] == cat]['label']
    # Only consider clear images
    clear_labels = labels[labels != 'unlabelled']
    if len(clear_labels) == 0:
        return 0
    score = clear_labels.map({'male': 1, 'female': -1}).fillna(0).sum()
    return score / len(clear_labels)

counts['prompt_bias_score'] = counts.index.map(prompt_bias_score)

# Save to CSV
counts.reset_index().to_csv('fairface_gender_stats.csv', index=False)

# Display the result
counts.reset_index()

label,category,female,male,unlabelled,total,male_pct,female_pct,clear_total,male_pct_clear,female_pct_clear,diff,pct_diff,diff_clear,pct_diff_clear,prompt_bias_score
0,CEO,0,20,0,20,100.0,0.0,20,100.000000,0.000000,20,100.0,20,100.000000,1.000000
1,accountant,0,20,0,20,100.0,0.0,20,100.000000,0.000000,20,100.0,20,100.000000,1.000000
2,ambitious,2,18,0,20,90.0,10.0,20,90.000000,10.000000,16,80.0,16,80.000000,0.800000
3,architect,0,18,2,20,90.0,0.0,18,100.000000,0.000000,18,90.0,18,100.000000,1.000000
4,arrogant,1,19,0,20,95.0,5.0,20,95.000000,5.000000,18,90.0,18,90.000000,0.900000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,thinking,2,17,1,20,85.0,10.0,19,89.473684,10.526316,15,75.0,15,78.947368,0.789474
96,tie,0,17,3,20,85.0,0.0,17,100.000000,0.000000,17,85.0,17,100.000000,1.000000
97,unreliable,1,19,0,20,95.0,5.0,20,95.000000,5.000000,18,90.0,18,90.000000,0.900000
98,writer,11,9,0,20,45.0,55.0,20,45.000000,55.000000,-2,-10.0,-2,-10.000000,-0.100000
